# Modelo 3: XGBoost
Entrenamiento del modelo supervisado XGBoost.

In [1]:
# CELDA 1: Cargar librerías y datos
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

X_train = np.load('../data/processed/X_train.npy')
y_train = np.load('../data/processed/y_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_test  = np.load('../data/processed/y_test.npy')

print('Train size:', X_train.shape)
print('Fraudes en train:', y_train.sum())

Train size: (331555, 12)
Fraudes en train: 30141


In [2]:
# CELDA 2: Calcular scale_pos_weight y configurar el modelo
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {ratio:.2f}')

modelo_xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=ratio,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

scale_pos_weight = 10.00


In [3]:
# CELDA 3: Entrenar el modelo
modelo_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)
print('XGBoost entrenado.')

[0]	validation_0-auc:0.99994


[50]	validation_0-auc:1.00000


[100]	validation_0-auc:1.00000


[150]	validation_0-auc:1.00000


[200]	validation_0-auc:1.00000


[250]	validation_0-auc:1.00000


... (2 outputs left out)

[400]	validation_0-auc:1.00000


[450]	validation_0-auc:1.00000


[499]	validation_0-auc:1.00000


XGBoost entrenado.


In [4]:
# CELDA 4: Evaluar preliminarmente
y_pred_xgb  = modelo_xgb.predict(X_test)
y_proba_xgb = modelo_xgb.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_xgb, target_names=['Legítima','Fraude']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba_xgb):.4f}')

              precision    recall  f1-score   support

    Legítima       1.00      1.00      1.00     64592
      Fraude       1.00      1.00      1.00      1232

    accuracy                           1.00     65824
   macro avg       1.00      1.00      1.00     65824
weighted avg       1.00      1.00      1.00     65824



AUC-ROC: 1.0000


In [5]:
# CELDA 5: Guardar el modelo
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(modelo_xgb, '../models/xgboost.pkl')
print('Modelo guardado: models/xgboost.pkl')

Modelo guardado: models/xgboost.pkl
